### Core Concepts 
#### Documents: This is LangChain's way of representing the piece of text with metadata (additional information like source,page no,date,etc). 


In [1]:
from langchain_core.documents import Document 

doc = Document(page_content="LangChain makes building LLM applications easier!",
               metadata={"source":"introduction.txt",
                         "author":"LangChain Team",
                         "date":"2025-01-15"})

print("Content:")
print(doc.page_content)
print(doc.metadata)
print(f"\nSource: {doc.metadata['source']}")

Content:
LangChain makes building LLM applications easier!
{'source': 'introduction.txt', 'author': 'LangChain Team', 'date': '2025-01-15'}

Source: introduction.txt


In [2]:
documents = [
    Document(
        page_content="Python is a high-level programming language.",
        metadata={"category": "programming", "difficulty": "beginner"}
    ),
    Document(
        page_content="Machine learning is a subset of artificial intelligence.",
        metadata={"category": "AI", "difficulty": "intermediate"}
    ),
    Document(
        page_content="RAG combines retrieval and generation for better LLM outputs.",
        metadata={"category": "AI", "difficulty": "advanced"}
    )
]

for i,doc in enumerate(documents,1):
    print(f"Document {i}")
    print(f"Content: {doc.page_content}")
    print(f"Category: {doc.metadata['category']}")
    print(f"Difficulty: {doc.metadata['difficulty']}")

Document 1
Content: Python is a high-level programming language.
Category: programming
Difficulty: beginner
Document 2
Content: Machine learning is a subset of artificial intelligence.
Category: AI
Difficulty: intermediate
Document 3
Content: RAG combines retrieval and generation for better LLM outputs.
Category: AI
Difficulty: advanced


### LCEL
This allows us to connect different components using pipe operator

In [3]:
from langchain_core.runnables import RunnableLambda 

def uppercase(text: str) -> str:
    return text.upper()

def  add_prefix(text: str) -> str:
    result = f"RESULT: {text}"
    print(f"Step 2: add_prefix -> {result}")
    return result

def add_emoji(text: str) -> str:
    result = f"✅ {text}"
    print(f"  Step 3: add_emoji → {result}")
    return result

uppercase_runnable = RunnableLambda(uppercase)
prefix_runnable = RunnableLambda(add_prefix)
emoji_runnable = RunnableLambda(add_emoji)

chain = uppercase_runnable  | prefix_runnable | emoji_runnable
chain.invoke("hello langchain")


Step 2: add_prefix -> RESULT: HELLO LANGCHAIN
  Step 3: add_emoji → ✅ RESULT: HELLO LANGCHAIN


'✅ RESULT: HELLO LANGCHAIN'

### LLM calls 

In [4]:
from langchain_groq import ChatGroq 

llm = ChatGroq(model="llama-3.1-8b-instant")
response = llm.invoke("What is LangChain explain in 2 sentences?")

In [5]:
response 

AIMessage(content='LangChain is an open-source framework for building conversational AI models, particularly focused on the development of multi-turn dialogue systems and chains of AI models that can interact and generate human-like conversations. It enables developers to build complex conversational workflows by connecting various AI models, such as chatbots, knowledge bases, and language generators, to create more sophisticated and context-aware conversational interfaces.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 45, 'total_tokens': 122, 'completion_time': 0.155375308, 'completion_tokens_details': None, 'prompt_time': 0.00233983, 'prompt_tokens_details': None, 'queue_time': 0.05509756, 'total_time': 0.157715138}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e57c6-5725-7341-a210-7a63

In [6]:
print("response type:",type(response))
print(f"content: {response.content}")
print(f"Response metadata:\n {response.response_metadata}")

# access response metadata
if "token_usage" in response.response_metadata:
    usage = response.response_metadata['token_usage']
    print(f"\nTokens Used:")
    print(f"  Prompt: {usage.get('prompt_tokens', 'N/A')}")
    print(f"  Completion: {usage.get('completion_tokens', 'N/A')}")
    print(f"  Total: {usage.get('total_tokens', 'N/A')}")

response type: <class 'langchain_core.messages.ai.AIMessage'>
content: LangChain is an open-source framework for building conversational AI models, particularly focused on the development of multi-turn dialogue systems and chains of AI models that can interact and generate human-like conversations. It enables developers to build complex conversational workflows by connecting various AI models, such as chatbots, knowledge bases, and language generators, to create more sophisticated and context-aware conversational interfaces.
Response metadata:
 {'token_usage': {'completion_tokens': 77, 'prompt_tokens': 45, 'total_tokens': 122, 'completion_time': 0.155375308, 'completion_tokens_details': None, 'prompt_time': 0.00233983, 'prompt_tokens_details': None, 'queue_time': 0.05509756, 'total_time': 0.157715138}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}

Tokens Use

### Instead of using plain strings use **prompt template** for better control 

In [7]:
from langchain_core.prompts import ChatPromptTemplate 

prompt = ChatPromptTemplate.from_template("Explain {topic} in simple terms suitable for beginners.")

print(prompt)
print(prompt.messages[0].prompt.template)

input_variables=['topic'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Explain {topic} in simple terms suitable for beginners.'), additional_kwargs={})]
Explain {topic} in simple terms suitable for beginners.


In [8]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

topics = ["machine learning","embeddings","vector databases"]

for topic in topics:
    print(f"\n{'='*60}")
    print(f"Topic: {topic.upper()}")
    print('='*60)

    explanation = chain.invoke({"topic":topic})
    print(explanation)


Topic: MACHINE LEARNING
**What is Machine Learning?**

Machine learning is a type of artificial intelligence (AI) that allows computers to learn from data without being explicitly programmed. It's like teaching a child to recognize objects, but instead of a child, it's a computer.

**How does it work?**

Imagine you have a big box of pictures of cats and dogs. You want the computer to be able to tell you which is which. Here's how machine learning works:

1. **Data collection**: You collect a bunch of pictures of cats and dogs.
2. **Training**: You feed these pictures to a computer program, and it starts to look for patterns. It might notice that cats have whiskers and dogs have floppy ears.
3. **Learning**: The computer program uses these patterns to make predictions. It might say, "Hey, this picture is probably a cat because it has whiskers."
4. **Testing**: You test the program with new, unseen pictures to see how well it's doing.

**Types of Machine Learning**

There are three typ

### Batch Processing
We can process multiple inputs efficiently

In [9]:
topics_batch = [{'topic':'RAG'},
                {"topic":"LCEL"},
                {"topic":"LangChain agents"}]

results = chain.batch(topics_batch)

for i, (input_dict, result) in enumerate(zip(topics_batch, results), 1):
    print(f"\n{i}. {input_dict['topic'].upper()}:")
    print(f"   {result[:100]}...")  # Print first 100 chars



1. RAG:
   RAG is a simple way to track progress or status of tasks, projects, or goals. It stands for:

1. **R...

2. LCEL:
   LCEL stands for Low-Carbon Economy Liquidity. It's a financial product that aims to help individuals...

3. LANGCHAIN AGENTS:
   **What are LangChain Agents?**

LangChain Agents are a type of AI system that can engage in conversa...
